# AI Gold Scalper - Cloud Training (Colab)
Train models in Google Colab and store artifacts in Google Cloud Storage.

In [ ]:
# 1. Clone repo and install deps
!git clone https://github.com/clayandthepotter/ai-gold-scalper.git
%cd ai-gold-scalper
!pip -q install -r requirements-core.txt

In [ ]:
# 2. Authenticate to Google Cloud
from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
client = storage.Client()
bucket_name = 'YOUR_BUCKET_NAME'  # <-- set this
bucket = client.bucket(bucket_name)
print('GCS ready:', bucket.name)

In [ ]:
# 3. Download training data from GCS (optional)
# If you already have data/market_data.csv locally, skip this.
import os
os.makedirs('data', exist_ok=True)
blob = bucket.blob('data/market_data.csv')
if blob.exists():
    blob.download_to_filename('data/market_data.csv')
    print('Downloaded market_data.csv')
else:
    print('No market_data.csv in bucket; trainer will use synthetic data')

In [ ]:
# 4. Train
import sys
from pathlib import Path
sys.path.insert(0, str(Path('scripts').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

from training.automated_model_trainer import AutomatedModelTrainer, TrainingConfig

cfg = TrainingConfig(min_samples=100, hyperopt_trials=5, cv_folds=3, validation_split=0.2)
trainer = AutomatedModelTrainer(cfg)
X, y = trainer._prepare_training_data()
print('Training data:', X.shape, y.shape)

from training.automated_model_trainer import ModelTrainer
ModelTrainer.optimize_hyperparameters = lambda self, X, y, config: self.default_params
mt = ModelTrainer('ridge')
res = mt.train(X.tail(5000), y.loc[X.tail(5000).index], cfg)
print('Training result:', res.success, res.validation_score, res.cross_val_score)

In [ ]:
# 5. Upload artifacts to GCS
import glob, os
for path in glob.glob('models/trained/*'):
    blob = bucket.blob(path)
    blob.upload_from_filename(path)
    print('Uploaded', path)
print('Done')